In [2]:
1+1

2

In [3]:
with open("../data/shakesphere.txt", 'r', encoding='utf-8') as f:
    text = f.read()

len(text)

1115394

In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)
print(''.join(chars))
print(VOCAB_SIZE)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hi there"))
print(decode(encode("hi there")))


[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [7]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
BLOCK_SIZE = 8
train_data[:BLOCK_SIZE+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:BLOCK_SIZE]
y = train_data[1:BLOCK_SIZE+1]

for i in range(BLOCK_SIZE):
    context = x[:i+1]
    target  = y[i]
    print(f"When input is {context} output is {target}")

When input is tensor([18]) output is 47
When input is tensor([18, 47]) output is 56
When input is tensor([18, 47, 56]) output is 57
When input is tensor([18, 47, 56, 57]) output is 58
When input is tensor([18, 47, 56, 57, 58]) output is 1
When input is tensor([18, 47, 56, 57, 58,  1]) output is 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) output is 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) output is 58


Lets create the Batch Loader function for batches

In [11]:
torch.manual_seed(1337)
BATCH_SIZE = 4 # how many independent sequences will we process in parallel
BLOCK_SIZE = 8 # maximum context length

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(data) - BLOCK_SIZE, (BATCH_SIZE, ))
    x = torch.stack([data[i: i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i+1: i+BLOCK_SIZE+1] for i in ix])
    return x, y
    
xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('-----------')

for b in range(BATCH_SIZE):
    for t in range(BLOCK_SIZE):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets: 
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----------
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is

In [12]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, VOCAB_SIZE) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(VOCAB_SIZE, VOCAB_SIZE)

    def forward(self, idx, target):
        logits = self.token_embedding_table(idx) #returns BTC 3 dim array
        B, T, C = logits.shape # Cause our logits were BTC and pytorch requires C to be the 2nd dim, so we just did B*T, C
        logits = logits.view(B*T, C)
        target = target.view(B*T) # Same with target
        loss = F.cross_entropy(logits, target)

        return logits, loss
    
model = BigramLanguageModel(VOCAB_SIZE)
logits, loss = model(xb, yb)
print(loss)


tensor(4.8786, grad_fn=<NllLossBackward0>)


In [13]:
xb.shape

torch.Size([4, 8])

Here, xb.shape is 4, 8 ie... 4 inputs per batch and each input has 8 tokens, in prod lingo, 4 is batch_size, 8 is context_length aka "time" and after embeddings, 
each token is represented into 65 numbers...ie channels..SO Batch_size, time, channels = (4,8,65)

In [14]:
-torch.log(torch.tensor(1/65))

tensor(4.1744)

Even if we wouldve guessed the next token uniformly, the negative log likelyhood wouldve been 4.17..So our initial guess is worse than uniformly guessing

In [15]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, VOCAB_SIZE) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(VOCAB_SIZE, VOCAB_SIZE)

    def forward(self, idx, target=None):
        logits = self.token_embedding_table(idx) #returns BTC 3 dim array

        if target is None:
            loss = None
        else:
            B, T, C = logits.shape # Cause our logits were BTC and pytorch requires C to be the 2nd dim, so we just did B*T, C
            logits = logits.view(B*T, C)
            target = target.view(B*T) # Same with target
            loss = F.cross_entropy(logits, target)

        return logits, loss
    
    def generate(self, idx, MAX_NEW_TOKENS):
        # idx is in (B*T) form of indices
        for _ in range(MAX_NEW_TOKENS):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # cause rightnow, logits has All batches, all time and all channels, we need only the last time
            probs = F.softmax(logits, dim=-1) # to get (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    
model = BigramLanguageModel(VOCAB_SIZE)
logits, loss = model(xb, yb)
print(loss)

idx = torch.zeros((1,1), dtype=torch.long)
MAX_NEW_TOKENS = 100

generated_text = decode(model.generate(idx, MAX_NEW_TOKENS)[0].tolist())
print(generated_text)

tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [16]:
print(generated_text)


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


Lets create a optimizer, Adam this time :)

In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [18]:
BATCH_SIZE = 32
EPOCHS = 10000


for steps in range(EPOCHS):
    xb,yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.382369041442871


Generating some text 

In [19]:
idx = torch.zeros((1,1), dtype=torch.long)
MAX_NEW_TOKENS = 300

generated_text = decode(model.generate(idx, MAX_NEW_TOKENS)[0].tolist())
print(generated_text)


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecor


Look at bigram.py for later updates, we will be shifting this to a python script and adding a few things to it as well

Bye Bye :)

I guess we are back

### Writing my 1st Self-Attention Block

Now we will write our first self-attention block

In [20]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

Till now, we looked at the last token in our bigram model, to predict the next token, What were we doing??

We had the text in string, we converted the whole text into numbers, not tensors but normal numbers by using the encoding {} dict..

Then we decided the context_length will be 8, so we created our dataset such that [w1, w2, ... w8] -> gives w8 and we did that

then each token got converted into embeddings of size 65, now each char/token can be represented using 65 tensor of numbers.
in generate, we take the input, we convert it into embeddings, and then take the lastt token, the [:, -1, :] step IMPORTANT

Then ofcourse we did softmax and then converted the numbers in the embedding into probabilities and then picked out a next token corresponding to the index using multinomial..

ie, while training, we are considering the context_length but while generating we are only looking at the last token...

This needs to change, so how to make the current token talk to previous n tokens??

Just how we took the -1 token, we can take avg(range(0,t-1) tokens) where t is the current token

In [21]:
# inefficient way of doing this

xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1, :] # (t, C) # give me all t's and all c's of the batch b
        xbow[b, t] = torch.mean(xprev, 0)

In [22]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [23]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

theres an efficint way to do this tho

In [24]:
a = torch.ones(3,3)
b = torch.rand((3,2))
c = a @ b
print(a)
print(b)
print(c)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.9062, 0.2759],
        [0.0258, 0.2173],
        [0.6806, 0.4048]])
tensor([[1.6125, 0.8980],
        [1.6125, 0.8980],
        [1.6125, 0.8980]])


Here, whats happening is for c[0,0], a's 0 row is getting dot multiplied with b's 0 row and so on...

but what if a is a triangular matrix

In [25]:
a = torch.tril(torch.ones(3,3))
b = torch.rand((3,2))
c = a @ b
print(a)
print(b)
print(c)

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
tensor([[0.0426, 0.4487],
        [0.3127, 0.8835],
        [0.2861, 0.2647]])
tensor([[0.0426, 0.4487],
        [0.3553, 1.3322],
        [0.6414, 1.5969]])


Now, C[0,0] is a's row dot producted with b 0 column, but since a 0 row only has one 1, 0.4185 and 0.9576 are ignored, same with c[0,1]

but in 2nd row of c, A has 2 1s...So they both are getting added and hence C 2nd row is nothing but sum of B 1st row and B 2nd row

If we normalize a, such that sum of all columns in a is 1...We can make C such that C is a moving average of previous rows..just how its moving sum

Putting this in code now

In [26]:
wei = torch.tril(torch.ones(T,T))
wei = wei/ wei.sum(1, keepdim=True)
xbow2 = wei @ x # (T, T) @ (B, T, C) -> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [27]:
xbow2[1], xbow[1]

(tensor([[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]]),
 tensor([[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]]))

Also we can do this

In [28]:
# instead of doing wei/sum of wei and stuff

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # puts -inf wherever tril has 0
wei = F.softmax(wei, dim=1) # softmax is also a kind of normalizer and softmax(-inf) = 0
xbow3 = wei @ x
torch.allclose(xbow2, xbow3)


True

Back to our script where we clean the code a bit up

So we added a positional embedding as well, where we have a array from 0 to batchSize-1, and then we tak the embeddings from a positional embedding table and then we add that embeddings to the main embeddings, so that each token also has a positional weight

Now lets study "self-attention"

In [29]:
torch.manual_seed(1337)
B, T, C = 4,8,32
x = torch.randn(B,T,C)

tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
out = wei@x
out.shape

torch.Size([4, 8, 32])

In [30]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

Rightnow, the weights are paying equal attention to all your previous tokens, but they shouldnt pay equal attention, cause some vowels might like consonents more and viceversa

Self-Attention helps with that, each token has 2 vectors, key and query...And we take dot products and choose the one which has the highest dot products

So lets add the self attention block

In [31]:
torch.manual_seed(1337)
B, T, C = 4,8,32
x = torch.randn(B,T,C)

#self attention block
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) -> (b, T, T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
v = value(x)
out = wei@v
out.shape

torch.Size([4, 8, 16])

Note

Lets convert the BatchNorm to Layer Norm and right now we were calculating the mean column wise, ie 0 now we, do it by rows

In [37]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1) -> None:
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # paramteres to be trained with backprop
        self.gamma = torch.ones(dim)
        self.beta = torch.randn(dim)

        # buffers for running mean and std
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x: torch.Tensor):
        if self.training:
            xmean = x.mean(1, keepdim=True)
            xvar = x.var(1, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_std
        
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum)* self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]
    
torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [38]:
x[:, 0].mean(), x[:, 0].std()

(tensor(0.3477), tensor(0.9482))

In [39]:
x[0, :].mean(), x[0, :].std()

(tensor(0.1086), tensor(1.4740))

We just changed the 0 to 1 in calculating the mean and std and layerNorm Tadaa

In [42]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5, momentum=0.1) -> None:
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # paramteres to be trained with backprop
        self.gamma = torch.ones(dim)
        self.beta = torch.randn(dim)

    def __call__(self, x: torch.Tensor):
        xmean = x.mean(1, keepdim=True)
        xvar = x.var(1, keepdim=True)
        
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        #we dont need to maintain any buffers, since we will always have enough data

        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]
    
torch.manual_seed(1337)
module = LayerNorm(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])